In [ ]:
import signal, time as _t
_T0 = _t.time()
def _cap(*a): raise TimeoutError("55 min cap hit after %.0fs" % (_t.time() - _T0))
signal.signal(signal.SIGALRM, _cap); signal.alarm(55 * 60)
import base64, io, zipfile, subprocess, sys, os, time, importlib
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.update({})
PIP = ["transformers>=5.17 gguf>=0.10 accelerate git+https://github.com/huggingface/diffusers"]
os.environ["SAM2_BUILD_CUDA"] = "0"
for _grp in PIP:
    _r = subprocess.run([sys.executable, "-m", "pip", "-q", "install"] + _grp.split(), capture_output=True, text=True)
    print("pip", _grp, "rc", _r.returncode, flush=True)
os.makedirs("/kaggle/working/code", exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode("..."))).extractall("/kaggle/working/code")
sys.path.insert(0, "/kaggle/working/code/prototype")
from layerproto.serve.profiles import PROFILES
prof = PROFILES["qwen21"]
_mod, _fn = prof.factory.split(":")
t_load = time.time()
dec = getattr(importlib.import_module(_mod), _fn)()
print("PROFILE LOADED", prof.name, round(time.time() - t_load), "s", flush=True)

import json, re, secrets, threading, urllib.request, uvicorn
from layerproto.remote_server import build_app
TOKEN = secrets.token_urlsafe(16)
stub = type("NoChain", (), {"info": lambda self: {"backend": "decompose-only", "profile": prof.name}})()
server = uvicorn.Server(uvicorn.Config(build_app(stub, TOKEN, decomposer=dec, profile=prof.name), host="127.0.0.1", port=8765, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
if not os.path.exists("/kaggle/working/cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/kaggle/working/cloudflared")
    os.chmod("/kaggle/working/cloudflared", 0o755)
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8765"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+[.]trycloudflare[.]com", line)
    if m:
        url = m.group(0); break
print("Put this into prototype/config.json on the laptop:")
print(json.dumps({"remote_url": url, "remote_token": TOKEN}, indent=1))

In [ ]:
# keep the session alive; stop this cell or the notebook to shut the backend down
while True:
    time.sleep(60)
